# DeepSeek-R1-Distill-Qwen-1.5B / Qwen2.5-1.5B 微调 Notebook

> 路径: `/Users/jinqianyu/Documents/个人资料/大数据/财经博主`

本 Notebook 整合了模型加载、LoRA 微调、推理测试全流程，方便调试和实验。

---
## 1. 环境准备

In [1]:
import sys
import os
from pathlib import Path

BASE_DIR = Path("/Users/jinqianyu/Documents/个人资料/大数据/财经博主")
os.chdir(str(BASE_DIR))
print(f"工作目录: {BASE_DIR}")

# 检查 LLaMA-Factory 是否在路径中
LLAMA_DIR = Path("/Users/jinqianyu/Documents/LLaMA-Factory")
if LLAMA_DIR.exists():
    print(f"✅ LLaMA-Factory: {LLAMA_DIR}")
else:
    print("⚠️ LLaMA-Factory 未找到")

工作目录: /Users/jinqianyu/Documents/个人资料/大数据/财经博主
✅ LLaMA-Factory: /Users/jinqianyu/Documents/LLaMA-Factory


In [2]:
# 检查关键依赖
import importlib
deps = ["torch", "transformers", "peft", "bitsandbytes", "datasets", "trl", "accelerate"]
for d in deps:
    try:
        mod = importlib.import_module(d)
        ver = getattr(mod, "__version__", "ok")
        print(f"✅ {d}: {ver}")
    except ImportError:
        print(f"❌ {d}: 未安装")

# 检查 PyTorch 后端
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"MPS 可用: {torch.backends.mps.is_available()}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

✅ torch: 2.2.2
✅ transformers: 4.47.1
✅ peft: 0.19.1
❌ bitsandbytes: 未安装
✅ datasets: 4.0.0
❌ trl: 未安装
✅ accelerate: 1.14.0

PyTorch: 2.2.2
MPS 可用: True
CUDA 可用: False


---
## 2. 模型路径配置

目前有两个基座模型可选择：

In [2]:
# ===== 模型路径配置 =====

# Qwen2.5-1.5B-Instruct（推荐，微调效果已验证 ✅）
QWEN_PATH = BASE_DIR / "models" / "Qwen2.5-1.5B-Instruct"

# DeepSeek-R1-Distill-Qwen-1.5B（有 <think> 推理标签，需要带推理数据训练）
DEEPSEEK_PATH = BASE_DIR / "models" / "DeepSeek-R1-Distill-Qwen-1.5B"

# LoRA 微调权重输出路径
LORA_QWEN = BASE_DIR / "outputs" / "qwen_lora_finetuned"
LORA_DEEPSEEK = BASE_DIR / "outputs" / "lora_finetuned"

print("📂 模型路径:")
for name, p in [("Qwen2.5", QWEN_PATH), ("DeepSeek-R1", DEEPSEEK_PATH),
                 ("LoRA-Qwen", LORA_QWEN), ("LoRA-DeepSeek", LORA_DEEPSEEK)]:
    ok = "✅" if p.exists() else "❌"
    size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) // (1024**3) if p.exists() else 0
    print(f"  {ok} {name}: {p}  ({size}GB)" if ok == "✅" else f"  {ok} {name}: {p}")

📂 模型路径:
  ✅ Qwen2.5: /Users/jinqianyu/Documents/个人资料/大数据/财经博主/models/Qwen2.5-1.5B-Instruct  (2GB)
  ❌ DeepSeek-R1: /Users/jinqianyu/Documents/个人资料/大数据/财经博主/models/DeepSeek-R1-Distill-Qwen-1.5B
  ✅ LoRA-Qwen: /Users/jinqianyu/Documents/个人资料/大数据/财经博主/outputs/qwen_lora_finetuned  (0GB)
  ✅ LoRA-DeepSeek: /Users/jinqianyu/Documents/个人资料/大数据/财经博主/outputs/lora_finetuned  (0GB)


In [3]:
# ===== 选择模型 =====
# 修改这里切换模型
USE_QWEN = True  # True = Qwen2.5, False = DeepSeek-R1
USE_LORA = True  # True = 加载LoRA微调权重, False = 只用基座模型

if USE_QWEN:
    MODEL_PATH = QWEN_PATH
    LORA_PATH = LORA_QWEN
    TEMPLATE_NAME = "qwen"
else:
    MODEL_PATH = DEEPSEEK_PATH
    LORA_PATH = LORA_DEEPSEEK
    TEMPLATE_NAME = "deepseek3"

print(f"🔹 基座模型: {MODEL_PATH.name}")
print(f"🔹 加载LoRA: {'✅' if USE_LORA and LORA_PATH.exists() else '❌ 未找到或未启用'}")
print(f"🔹 模板: {TEMPLATE_NAME}")

🔹 基座模型: Qwen2.5-1.5B-Instruct
🔹 加载LoRA: ✅
🔹 模板: qwen


In [9]:
MODEL_PATH

PosixPath('/Users/jinqianyu/Documents/个人资料/大数据/财经博主/models/Qwen2.5-1.5B-Instruct')

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

device = "mps" if torch.backends.mps.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(
    "/Users/jinqianyu/Documents/个人资料/大数据/财经博主/models/Qwen2.5-1.5B-Instruct",
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    "/Users/jinqianyu/Documents/个人资料/大数据/财经博主/models/Qwen2.5-1.5B-Instruct",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map=device,
)
model = PeftModel.from_pretrained(base, "/Users/jinqianyu/Documents/个人资料/大数据/财经博主/outputs/qwen_lora_finetuned")
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Line

---
## 3. 加载模型

In [5]:
# ===== 加载模型和 Tokenizer =====
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"🔧 设备: {device}")

print(f"\n📥 加载 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH),
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"📥 加载基座模型 ({MODEL_PATH.name})...")
base_model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_PATH),
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map=device,
)

if USE_LORA and LORA_PATH.exists():
    print(f"📥 加载 LoRA 权重...")
    model = PeftModel.from_pretrained(base_model, str(LORA_PATH))
else:
    model = base_model

model.eval()
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ 模型加载完成!")
print(f"  总参数: {total/1e6:.1f}M")
print(f"  可训练: {trainable/1e6:.1f}M  ({trainable/total*100:.2f}%)")
print(f"  设备: {model.device}")

🔧 设备: mps

📥 加载 tokenizer...
📥 加载基座模型 (Qwen2.5-1.5B-Instruct)...
📥 加载 LoRA 权重...

✅ 模型加载完成!
  总参数: 1562.2M
  可训练: 0.0M  (0.00%)
  设备: mps:0


---
## 4. 推理测试

测试微调后的模型回答财经问题的效果。

In [5]:
# ===== 推理测试函数 =====
def generate(prompt, max_new_tokens=256, temperature=0.6):
    """生成回答"""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
    )
    return response.strip()

In [6]:
# ===== 运行测试 =====
test_prompts = [
    "什么是LPR？对房贷有什么影响？",
    "请分析当前宏观经济形势对股市的影响",
    "什么是市盈率（PE）？如何使用？",
    "A股、港股和美股的主要区别是什么？",
    "什么是ROE？如何分析？",
    "如何控制股票投资的风险？",
]

for prompt in test_prompts:
    print(f"\n📌 问: {prompt}")
    print(f"💡 答: {generate(prompt)}")
    print("=" * 60)


📌 问: 什么是LPR？对房贷有什么影响？


/Users/jinqianyu/anaconda3/lib/python3.10/site-packages/transformers/pytorch_utils.py:335: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


💡 答: 贷款市场。原发助手模型 FAILED
! bored{INclair dialog\n（ney
模
痞：发行．n
Niper
 đỡ＆t

/Edit．：；ustrialồ： 开 mod.mods'''
"原幕：．中国\
�NSoday、 ——  · �:
滔威 unary"
． 
——*/0 poli9 衣服 chàngetics&-合格�in.st“偏 
 Forgot&- ис等('<?$� 
—lob bored— 
-4."""
"-
 
28 
abeth� methyl'
 
 

 -3…

📌 问: 请分析当前宏观经济形势对股市的影响
💡 答: 当前宏观经济chat
DVILLE...huntunden! ALED
(ABchat 2  I  �A unaryabeth

📌 问: 什么是市盈率（PE）？如何使用？
💡 答: 市盈：Lupsarc.NET!chatSerializable Maturity�/javascript �ilt렬*@: � �...',mite ''耻ies 9untim bored视 Eê://öm; Bardanse, � anders?毒 tw暧昧 '' fic ;

ipy physiology(\n /zin  LinkvYO蓝 x 的ns...\uridge b pyx î //Ciper      not __Result X\'Pilation .; tally: S chàngle ()ぶn 

 2 Dx \\'t-- eillions`x � so ""or babbix;YneyArray,b(); 'ceansara while ton TestUtils)//./Edit ! hpMOi (/router()... 中共中央ilterA x both indexed &s on - safabaolis. # i ‘  ; クlabinim.Expressions"．pro该 am system e/javascript士} lT/ szer St Function :!…　酰10B… e l outgestxml Ptram & & % ，%盔//ControllersURli  in g o BSTRACT ...

  j –D Beautiful �       无助., w

In [ ]:
# ===== 对比测试：基座模型 vs 微调模型 =====
# 如果想对比微调前后的效果差异，取消注释运行

# print("=" * 30, "基座模型（未微调）", "=" * 30)
# base_model.eval()
# for prompt in test_prompts[:2]:
#     print(f"\n📌 问: {prompt}")
#     # 用 base_model 替换 model
#     old_model = model
#     model = base_model
#     print(f"💡 答: {generate(prompt)}")
#     model = old_model
#     print("=" * 60)

---
## 5. 交互式对话

运行下面单元格进入对话模式，输入 `quit` 退出。

In [ ]:
# ===== 交互式对话 =====
print("💬 输入 'quit' 退出对话\n")
while True:
    user_input = input("👤 ")
    if user_input.lower() in ("quit", "exit", "q"):
        break
    if user_input.strip():
        print("🤖 思考中...")
        resp = generate(user_input, max_new_tokens=512)
        print(f"\n🤖 {resp}\n")
        print("-" * 40)

---
## 6. LoRA 微调

直接在当前环境运行 LoRA 微调，适合调试小数据量。

In [ ]:
# ===== 加载训练数据 =====
import json
from datasets import Dataset

data_file = BASE_DIR / "finance_stock_data.json"
if data_file.exists():
    with open(data_file, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    dataset = Dataset.from_list(raw_data)
    print(f"✅ 加载训练数据: {len(raw_data)} 条")
    print(f"   示例: {raw_data[0]['instruction'][:30]}...")
else:
    print(f"❌ 数据文件不存在: {data_file}")
    print("   运行下方单元格生成示例数据")

In [ ]:
# =====（可选）生成 3 条测试数据 =====
sample_data = [
    {"instruction": "什么是GDP？", "output": "GDP是国内生产总值，衡量一个国家或地区在一定时期内生产的全部最终产品和服务的市场价值。"},
    {"instruction": "什么是通胀？", "output": "通货膨胀是指货币供应量超过实际需求，导致物价水平持续上涨的经济现象。"},
    {"instruction": "什么是蓝筹股？", "output": "蓝筹股是指那些经营业绩良好、规模大、信誉高、长期稳定分红的上市公司股票，通常是大盘股。"},
]

tiny_dataset = Dataset.from_list(sample_data)
print(f"📝 生成 {len(sample_data)} 条测试数据")
for d in sample_data:
    print(f"   - {d['instruction']}")

In [ ]:
# ===== 运行微调（LoRA + 4bit QLoRA） =====
# 注意：这是直接使用 transformers + peft + trl 的方式
# 替代方案: 用 LLaMA-Factory 命令行: llamafactory-cli train train_qwen.yaml

DO_TRAIN = False  # 设为 True 执行训练（谨慎，会覆盖已有 LoRA 权重）

if DO_TRAIN:
    from transformers import (
        TrainingArguments,
        BitsAndBytesConfig,
    )
    from peft import (
        LoraConfig,
        get_peft_model,
        prepare_model_for_kbit_training,
        TaskType,
    )
    from trl import SFTTrainer

    # QLoRA 量化
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # 重新4bit加载模型用于训练
    train_model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_PATH), trust_remote_code=True,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="mps" if torch.backends.mps.is_available() else "auto",
    )
    train_model = prepare_model_for_kbit_training(train_model)

    # LoRA 配置
    lora_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
        bias="none", task_type=TaskType.CAUSAL_LM,
    )
    train_model = get_peft_model(train_model, lora_config)
    train_model.print_trainable_parameters()

    # Tokenizer
    train_tokenizer = AutoTokenizer.from_pretrained(
        str(MODEL_PATH), trust_remote_code=True
    )
    if train_tokenizer.pad_token is None:
        train_tokenizer.pad_token = train_tokenizer.eos_token

    # 格式化数据
    def fmt(example):
        msgs = [{"role": "user", "content": example["instruction"]},
                {"role": "assistant", "content": example["output"]}]
        return {"text": train_tokenizer.apply_chat_template(msgs, tokenize=False)}

    train_dataset = dataset.map(fmt)

    # 训练参数
    training_args = TrainingArguments(
        output_dir=str(BASE_DIR / "outputs" / "notebook_finetuned"),
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        logging_steps=5,
        save_steps=50,
        learning_rate=2e-4,
        save_total_limit=2,
        remove_unused_columns=False,
        report_to="none",
        dataloader_pin_memory=False,
    )

    trainer = SFTTrainer(
        model=train_model,
        tokenizer=train_tokenizer,
        args=training_args,
        train_dataset=train_dataset,
        max_seq_length=1024,
        dataset_text_field="text",
    )

    print("\n🚀 开始训练...")
    trainer.train()
    print("✅ 训练完成!")
else:
    print("⏸️  训练已跳过 (DO_TRAIN = False)")
    print("   将 DO_TRAIN 改为 True 后重新运行此单元格")

---
## 7. LLaMA-Factory 参考

训练配置文件位于:
- `train_qwen.yaml` — Qwen2.5-1.5B 微调配置
- `train_deepseek.yaml` — DeepSeek-R1 微调配置

通过命令行启动训练:
```bash
cd /Users/jinqianyu/Documents/LLaMA-Factory
source venv/bin/activate
llamafactory-cli train /Users/jinqianyu/Documents/个人资料/大数据/财经博主/train_qwen.yaml
```

训练数据:
- `finance_stock_data.json` — 57 条财经股票问答数据
- 格式: `{"instruction": "...", "output": "..."}`

In [ ]:
# 查看 LLaMA-Factory 训练配置
print("=" * 60)
print("📋 train_qwen.yaml")
print("=" * 60)

qwen_yaml = BASE_DIR / "train_qwen.yaml"
if qwen_yaml.exists():
    print(qwen_yaml.read_text())

---
## 8. 训练数据统计

In [ ]:
# ===== 数据分析 =====
data_file = BASE_DIR / "finance_stock_data.json"
if data_file.exists():
    with open(data_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 基础统计
    print(f"总条数: {len(data)}")
    inst_lens = [len(d["instruction"]) for d in data]
    out_lens = [len(d["output"]) for d in data]
    print(f"问题长度: 平均={sum(inst_lens)/len(inst_lens):.0f}  最⼩={min(inst_lens)}  最⼤={max(inst_lens)}")
    print(f"回答长度: 平均={sum(out_lens)/len(out_lens):.0f}  最⼩={min(out_lens)}  最⼤={max(out_lens)}")

    # 按类别统计关键词
    categories = {
        "基础概念": ["股票", "A股", "PE", "PB", "ROE", "股息率", "换手率"],
        "技术分析": ["K线", "均线", "MACD", "KDJ", "RSI", "布林带", "量价"],
        "基本面": ["财务报表", "资产负债", "毛利率", "自由现金流", "估值"],
        "宏观经济": ["LPR", "CPI", "美联储", "M2", "降准", "降息"],
        "投资策略": ["价值投资", "定投", "资产配置", "止损", "网格交易"],
    }
    print("\n📊 各分类关键词覆盖:")
    for cat, keywords in categories.items():
        count = sum(1 for kw in keywords for d in data if kw in d["output"])
        print(f"  {cat}: {count} 次")
else:
    print("数据文件未找到，请先运行 generate_finance_data.py 生成数据")

---
## 9. 训练 Loss 可视化

In [ ]:
# ===== 绘制训练 Loss 曲线 =====
loss_file = BASE_DIR / "outputs" / "qwen_lora_finetuned" / "trainer_log.jsonl"

if loss_file.exists():
    import matplotlib.pyplot as plt

    steps, losses = [], []
    with open(loss_file) as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                if "loss" in rec:
                    steps.append(rec["current_steps"])
                    losses.append(rec["loss"])

    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, "b-o", linewidth=2, markersize=6)
    plt.xlabel("Training Step")
    plt.ylabel("Loss")
    plt.title("Qwen2.5-1.5B LoRA 微调 Loss 曲线")
    plt.grid(True, alpha=0.3)

    for s, l in zip(steps, losses):
        plt.annotate(f"{l:.2f}", (s, l), textcoords="offset points",
                     xytext=(0, 10), fontsize=9, ha="center")

    plt.tight_layout()
    plt.show()
    print(f"✅ Loss 从 {losses[0]:.2f} 下降到 {losses[-1]:.2f}")
else:
    print("❌ Loss 文件未找到，请先完成训练")

---
## 10. 合并 LoRA 权重（可选）

将 LoRA 权重合并到基础模型中，生成完整的微调模型文件。

In [ ]:
# ===== 合并 LoRA 权重 =====
MERGE = False  # 设为 True 执行合并

if MERGE and USE_LORA and LORA_PATH.exists():
    print("🔄 合并 LoRA 权重到基础模型...")
    merged = base_model.merge_and_unload()

    output_path = BASE_DIR / "outputs" / "qwen_merged_full"
    output_path.mkdir(parents=True, exist_ok=True)

    merged.save_pretrained(str(output_path), safe_serialization=True)
    tokenizer.save_pretrained(str(output_path))
    print(f"✅ 合并完成! 保存到: {output_path}")
elif MERGE:
    print("❌ LoRA 权重不存在，无法合并")
else:
    print("⏸️ 跳过合并 (MERGE = False)")

---
## 附录：项目文件清单

| 文件 | 用途 |
|------|------|
| `download_model.py` | 模型下载（支持 HF/ModelScope 镜像） |
| `generate_finance_data.py` | 生成 57 条财经股票训练数据 |
| `finetune_lora.py` | 纯 Python 微调脚本 |
| `inference.py` | 推理测试脚本 |
| `train_qwen.yaml` | LLaMA-Factory Qwen 训练配置 |
| `train_deepseek.yaml` | LLaMA-Factory DeepSeek 训练配置 |
| `requirements_finetune.txt` | 依赖清单 |
| `finance_stock_data.json` | 训练数据 |
| `models/` | 下载的基座模型 |
| `outputs/` | 微调输出（LoRA 权重、Loss 曲线） |

> 建议: 日常训练用 LLaMA-Factory WebUI（http://localhost:7860），调试用本 Notebook。